# 陈小群战法 - 选股与策略执行

> **策略来源**: 陈小群游资战法  
> **可靠性评级**: B级（中高可靠性）  
> **更新时间**: 2026-01-14

---

## 📊 功能说明

本Notebook用于根据市场情绪周期执行相应的选股策略，是陈小群战法的核心执行步骤。

### 策略流程

| 情绪周期 | 策略 | 仓位 | 选股方法 |
|---------|------|------|---------|
| **退潮期** | 空仓等待 | 0% | 不选股 |
| **启动期** | 首板卡位术 | 10% | 早盘9:35前涨停，流通市值<30亿 |
| **加速期** | 龙头战法 | 50%+ | 识别市场总龙头，重仓持有 |
| **过热期** | 逐步减仓 | 30-50% | 持有现有仓位，准备退出 |

### 选股条件（首板卡位术）

1. ✅ **早盘9:35前涨停**：资金积极性强
2. ✅ **流通市值<30亿**：易于拉升
3. ✅ **封单量>流通市值2%**：资金共识强
4. ✅ **题材新颖、有想象空间**：容易形成热点
5. ✅ **板块内至少3只跟风涨停**：形成板块效应

### 二板定龙术确认条件

1. ✅ **换手率>25%**：充分换手，新资金入场
2. ✅ **分时走势**：急跌不破开盘价，反弹带量拉升
3. ✅ **板块内至少3只跟风股涨停**：形成梯队效应
4. ✅ **确认龙头地位**：板块内涨幅最大或最早涨停

---

## 🔧 环境初始化

In [1]:
# 设置输出默认可滚动（限制最大高度）
from IPython.display import HTML, display

display(HTML("""
<style>
    .jp-OutputArea-output {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    .jp-Cell-outputArea {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    .jp-OutputArea-child {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
</style>
"""))

print("✅ 输出区域已设置为可滚动（最大高度600px）")

✅ 输出区域已设置为可滚动（最大高度600px）


In [2]:
import sys
from pathlib import Path

# 自动检测项目根目录
current_dir = Path.cwd()
project_root = None
for parent in [current_dir] + list(current_dir.parents):
    if (parent / 'core').exists() and (parent / 'config').exists():
        project_root = parent
        break

if project_root is None:
    project_root = Path('/home/taotao/.cursor/worktrees/TRQuant/ope')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 使用统一环境初始化
from notebooks.lib import setup_research_environment
env = setup_research_environment(verbose=True)

2026-01-14 08:21:09,969 - notebooks.lib.research_init - INFO - ✅ 项目根目录: /home/taotao/.cursor/worktrees/TRQuant/ope
2026-01-14 08:21:09,974 - notebooks.lib.research_init - INFO - ✅ 加载配置: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/research.yaml


研究环境状态
项目根目录: /home/taotao/.cursor/worktrees/TRQuant/ope
Python 版本: 3.12.3
当前时间: 2026-01-14 08:21:09
JQData 客户端: ⏳ 未初始化
趋势分析器: ⏳ 未初始化
评估引擎: ⏳ 未初始化


## 📊 1. 读取第一步的情绪周期判断结果

In [3]:
# 从第一步notebook获取情绪周期判断结果
# 如果第一步已运行，可以直接使用结果
# 否则需要先运行第一步notebook

print("=" * 80)
print("📊 读取第一步的情绪周期判断结果")
print("=" * 80)

# 尝试从第一步的结果中读取
emotion_cycle = None
position = None
strategy = None
limit_up_count = None
max_height = None
zhaban_rate = None

# 方法1: 尝试从全局变量中读取（如果第一步在同一kernel中运行）
try:
    if 'result' in globals() and isinstance(result, dict):
        emotion_cycle = result.get('cycle', None)
        position = result.get('position', None)
        strategy = result.get('strategy', None)
        limit_up_count = result.get('limit_up_count', None)
        max_height = result.get('max_height', None)
        zhaban_rate = result.get('zhaban_rate', None)
        print(f"✅ 从第一步读取到结果（全局变量）:")
        print(f"   情绪周期: {emotion_cycle}")
        print(f"   建议仓位: {position}")
        print(f"   推荐策略: {strategy}")
        if limit_up_count is not None:
            print(f"   涨停家数: {limit_up_count}只")
        if max_height is not None:
            print(f"   最高连板: {max_height}板")
        if zhaban_rate is not None:
            print(f"   炸板率: {zhaban_rate:.2f}%")
except NameError:
    pass

# 方法2: 如果方法1失败，尝试从locals()中读取
if emotion_cycle is None:
    try:
        if 'result' in locals() and isinstance(result, dict):
            emotion_cycle = result.get('cycle', None)
            position = result.get('position', None)
            strategy = result.get('strategy', None)
            limit_up_count = result.get('limit_up_count', None)
            max_height = result.get('max_height', None)
            zhaban_rate = result.get('zhaban_rate', None)
            print(f"✅ 从第一步读取到结果（局部变量）:")
            print(f"   情绪周期: {emotion_cycle}")
            print(f"   建议仓位: {position}")
            print(f"   推荐策略: {strategy}")
    except:
        pass

# 方法3: 如果都失败，提示用户手动设置或运行第一步
if emotion_cycle is None:
    print(f"⚠️  未找到第一步的结果变量 'result'")
    print(f"\n💡 解决方案（三选一）:")
    print(f"   方案1: 先运行 01_market_environment_judgment.ipynb（推荐）")
    print(f"   方案2: 手动设置以下变量（在当前cell下方添加）:")
    print(f"      emotion_cycle = '启动期'  # 退潮期/启动期/加速期/过热期")
    print(f"      position = '10%'  # 0%/10%/50%+/30-50%")
    print(f"      strategy = '首板卡位术'  # 空仓等待/首板卡位术/龙头战法/逐步减仓")
    print(f"   方案3: 使用默认值继续（当前将使用默认值）")
    
    # 设置默认值（用于测试）
    print(f"\n⚠️  使用默认值进行测试（建议先运行第一步）")
    emotion_cycle = "启动期"  # 默认值
    position = "10%"
    strategy = "首板卡位术"
    print(f"   默认情绪周期: {emotion_cycle}")
    print(f"   默认仓位: {position}")
    print(f"   默认策略: {strategy}")

2026-01-14 08:21:13,540 - core.notebook_result_manager - INFO - MongoDB索引创建成功
2026-01-14 08:21:13,540 - core.notebook_result_manager - INFO - MongoDB连接成功: jqquant
2026-01-14 08:21:13,540 - core.notebook_result_manager - INFO - NotebookResultManager 初始化: chen_xiaoqun_strategy/01_market_environment_judgment
2026-01-14 08:21:13,540 - core.notebook_result_manager - INFO - 输出目录: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/01_market_environment_judgment


📊 读取第一步的情绪周期判断结果
✅ 从MongoDB读取到第一步的最新结果（运行ID: 20260114_211007）:
   运行日期: 2026-01-14 21:10:07
   情绪周期: 过热期
   建议仓位: 30-50%
   推荐策略: 逐步减仓
   涨停家数: 102只
   最高连板: 5板
   炸板率: 4.67%



## 🎯 2. 首板卡位术（启动期，10%试错仓）

In [4]:
"""
首板卡位术 - 启动期选股策略
选股条件：
1. 早盘9:35前涨停
2. 流通市值<30亿
3. 封单量>流通市值2%
4. 题材新颖、有想象空间
5. 板块内至少3只跟风涨停
"""

import akshare as ak
import pandas as pd
from datetime import datetime, timezone, timedelta
import numpy as np

def cn_today_str():
    """获取中国时间（UTC+8）的当前日期字符串（标准格式YYYY-MM-DD）"""
    cn_now = datetime.now(timezone.utc) + timedelta(hours=8)
    return cn_now.strftime('%Y-%m-%d')

def cn_today_str_compact():
    """获取中国时间（UTC+8）的当前日期字符串（紧凑格式YYYYMMDD，用于AKShare接口）"""
    cn_now = datetime.now(timezone.utc) + timedelta(hours=8)
    return cn_now.strftime('%Y%m%d')

def cn_now():
    """获取中国时间（UTC+8）的当前时间"""
    return datetime.now(timezone.utc) + timedelta(hours=8)

print("=" * 80)
print("🎯 首板卡位术 - 启动期选股策略")
print("=" * 80)

# 检查情绪周期，只有启动期才执行首板卡位术
if 'emotion_cycle' in globals() and emotion_cycle is not None:
    print(f"\n📊 当前情绪周期: {emotion_cycle}")
    print(f"   建议仓位: {position}")
    print(f"   推荐策略: {strategy}")
    
    if emotion_cycle == "退潮期":
        print(f"\n⚠️  当前处于退潮期，不建议执行首板卡位术")
        print(f"   💡 建议：空仓等待，不操作")
    elif emotion_cycle == "启动期":
        print(f"\n✅ 当前处于启动期，可以执行首板卡位术（10%试错仓）")
    elif emotion_cycle == "加速期":
        print(f"\n⚠️  当前处于加速期，建议使用龙头战法（50%+重仓）")
        print(f"   💡 首板卡位术不适用于加速期")
    elif emotion_cycle == "过热期":
        print(f"\n⚠️  当前处于过热期，建议逐步减仓（30-50%）")
        print(f"   💡 首板卡位术不适用于过热期")
else:
    print(f"\n⚠️  未读取到情绪周期，使用默认策略（启动期）")

# 获取当前时间
cn_time = cn_now()
today_str = cn_today_str()
current_hour = cn_time.hour
current_minute = cn_time.minute

print(f"\n📅 当前时间: {cn_time.strftime('%Y-%m-%d %H:%M:%S')} (中国时区)")
print(f"   日期: {today_str}")

# 检查是否在交易时间内
if current_hour < 9 or (current_hour == 9 and current_minute < 30):
    print(f"\n⚠️  当前时间早于9:30，无法获取实时涨停数据")
    print(f"   💡 建议在9:30-15:00之间运行此notebook")
elif current_hour >= 15:
    print(f"\n⚠️  当前时间已收盘，将获取当日历史数据")
else:
    print(f"\n✅ 当前在交易时间内，可以获取实时数据")

# 获取涨停板数据
print(f"\n📊 正在获取涨停板数据...")
# 注意：AKShare接口需要紧凑格式（YYYYMMDD），不是标准格式（YYYY-MM-DD）
today_compact = cn_today_str_compact()

# 可以指定历史日期进行测试（例如：2026-01-13）
# 如果想使用特定日期，取消下面的注释并修改日期
# test_date = "20260113"  # 格式：YYYYMMDD
# limit_up_data = ak.stock_zt_pool_em(date=test_date)

try:
    # 先尝试不指定日期（获取最新数据）
    limit_up_data = ak.stock_zt_pool_em()
    data_date = "最新交易日"
    if limit_up_data is None or limit_up_data.empty:
        # 如果最新数据为空，尝试指定日期（紧凑格式）
        limit_up_data = ak.stock_zt_pool_em(date=today_compact)
        data_date = today_str
    if limit_up_data is not None and not limit_up_data.empty:
        print(f"✅ 获取成功，共 {len(limit_up_data)} 只涨停股票")
        print(f"📅 数据日期: {data_date}")
        
        # 显示列名
        print(f"\n📋 数据列: {list(limit_up_data.columns)}")
        
        # 显示前5条数据
        print(f"\n📊 前5只涨停股票:")
        print(limit_up_data.head(5).to_string())
    else:
        print(f"⚠️  返回数据为空")
        limit_up_data = None
except Exception as e:
    print(f"❌ 获取失败: {str(e)[:150]}")
    import traceback
    traceback.print_exc()
    limit_up_data = None

/home/taotao/.cursor/worktrees/TRQuant/ope/venv/lib/python3.12/site-packages/py_mini_racer/py_mini_racer.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


🎯 首板卡位术 - 启动期选股策略

📊 当前情绪周期: 过热期
   建议仓位: 30-50%
   推荐策略: 逐步减仓

⚠️  当前处于过热期，建议逐步减仓（30-50%）
   💡 首板卡位术不适用于过热期

📅 当前时间: 2026-01-14 21:21:18 (中国时区)
   日期: 2026-01-14

⚠️  当前时间已收盘，将获取当日历史数据

📊 正在获取涨停板数据...
✅ 获取成功，共 102 只涨停股票
📅 数据日期: 2026-01-14

📋 数据列: ['序号', '代码', '名称', '涨跌幅', '最新价', '成交额', '流通市值', '总市值', '换手率', '封板资金', '首次封板时间', '最后封板时间', '炸板次数', '涨停统计', '连板数', '所属行业']

📊 前5只涨停股票:
   序号      代码    名称        涨跌幅    最新价         成交额          流通市值           总市值       换手率        封板资金  首次封板时间  最后封板时间  炸板次数 涨停统计  连板数  所属行业
0   1  002044  美年健康   9.986505   8.15  2606516992  3.157496e+10  3.190117e+10  8.257297   681189982  092500  093400     1  4/4    4  医疗服务
1   2  002112  三变科技  10.011442  19.23   235220418  5.039790e+09  5.656145e+09  4.667266   263850003  092500  092500     0  6/3    2  电网设备
2   3  002115  三维通信   9.994386  19.59   148909761  1.473551e+10  1.588732e+10  1.010550  1570949232  092500  092500     0  4/4    4  互联网服
3   4  002153  石基信息   9.970016  14.67   252494641  2.346854e+10  4.

In [5]:
"""
筛选首板卡位术候选股票
条件：
1. 早盘9:35前涨停（需要分时数据验证）
2. 流通市值<30亿
3. 封单量>流通市值2%
4. 板块内至少3只跟风涨停
"""

if limit_up_data is not None and not limit_up_data.empty:
    print("=" * 80)
    print("🔍 筛选首板卡位术候选股票")
    print("=" * 80)
    
    # 检查是否应该执行筛选（根据情绪周期）
    should_filter = True
    if 'emotion_cycle' in globals() and emotion_cycle is not None:
        if emotion_cycle != "启动期":
            print(f"\n⚠️  当前情绪周期为 {emotion_cycle}，首板卡位术不适用")
            print(f"   💡 建议：")
            if emotion_cycle == "退潮期":
                print(f"      • 空仓等待，不操作")
            elif emotion_cycle == "加速期":
                print(f"      • 使用龙头战法（50%+重仓）")
            elif emotion_cycle == "过热期":
                print(f"      • 逐步减仓（30-50%）")
            should_filter = False
        else:
            print(f"\n✅ 当前情绪周期为启动期，执行首板卡位术筛选")
    
    if not should_filter:
        print(f"\n⚠️  跳过首板卡位术筛选")
        first_board_candidates = pd.DataFrame()
    else:
        # 筛选条件
        candidates = []
        
        for idx, row in limit_up_data.iterrows():
            stock_info = {}
            
            # 条件1: 流通市值<30亿（必须满足）
            if '流通市值' not in limit_up_data.columns:
                continue  # 如果没有流通市值字段，跳过
            
            market_cap = row['流通市值']
            if pd.isna(market_cap):
                continue  # 流通市值为空，跳过
            
            if market_cap >= 30 * 1e8:  # 30亿
                continue  # 流通市值>=30亿，跳过
            
            stock_info['流通市值(亿)'] = market_cap / 1e8
            
            # 条件2: 封板资金>流通市值2%（必须满足）
            # 注意：AKShare返回的字段是'封板资金'，不是'封单额'
            if '封板资金' not in limit_up_data.columns:
                continue  # 如果没有封板资金字段，跳过
            
            limit_amount = row['封板资金']
            if pd.isna(limit_amount) or limit_amount == 0:
                continue  # 封板资金为空或0，跳过
            
            # 必须有流通市值才能计算占比
            if '流通市值(亿)' not in stock_info or stock_info['流通市值(亿)'] is None:
                continue  # 如果没有流通市值，跳过
            
            # 计算封板资金占比
            limit_ratio = limit_amount / (stock_info['流通市值(亿)'] * 1e8)
            if limit_ratio < 0.02:  # 必须>=2%
                continue  # 封板资金占比不足2%，跳过
            
            stock_info['封板资金占比(%)'] = limit_ratio * 100
            
            # 保存股票信息
            stock_info['代码'] = row.get('代码', '')
            stock_info['名称'] = row.get('名称', '')
            stock_info['首次封板时间'] = row.get('首次封板时间', '')
            stock_info['连板数'] = row.get('连板数', 1)
            stock_info['涨跌幅'] = row.get('涨跌幅', 0)
            stock_info['换手率'] = row.get('换手率', 0)
            stock_info['所属行业'] = row.get('所属行业', '')
            if '封板资金' in limit_up_data.columns:
                stock_info['封板资金(万)'] = row.get('封板资金', 0) / 1e4
            
            # 只选择首板（连板数=1）
            if stock_info.get('连板数', 1) == 1:
                candidates.append(stock_info)
        
        print(f"\n✅ 筛选完成，找到 {len(candidates)} 只首板候选股票")
        print(f"\n📋 筛选条件（所有条件必须同时满足）:")
        print(f"   1. ✅ 连板数 = 1（首板）")
        print(f"   2. ✅ 流通市值 < 30亿")
        print(f"   3. ✅ 封板资金占比 >= 2%")
        print(f"\n💡 说明：只有同时满足以上3个条件的股票才会被选中")
        
        if len(candidates) > 0:
            # 转换为DataFrame显示
            candidates_df = pd.DataFrame(candidates)
            
            # 按封板资金占比排序（从高到低）
            if '封板资金占比(%)' in candidates_df.columns:
                candidates_df = candidates_df.sort_values('封板资金占比(%)', ascending=False)
            
            print(f"\n" + "=" * 80)
            print(f"📊 首板候选股票列表（共 {len(candidates_df)} 只）")
            print("=" * 80)
            print(f"💡 说明：已按封板资金占比从高到低排序")
            print()
            
            # 显示主要信息表格
            display_cols = ['代码', '名称', '流通市值(亿)', '封板资金占比(%)', '首次封板时间', '换手率', '所属行业']
            available_cols = [c for c in display_cols if c in candidates_df.columns]
            
            # 使用更清晰的格式显示
            print("┌" + "─" * 78 + "┐")
            print("│" + " " * 20 + "首板候选股票列表" + " " * 40 + "│")
            print("├" + "─" * 78 + "┤")
            print(candidates_df[available_cols].to_string(index=False))
            print("└" + "─" * 78 + "┘")
            
            # 显示前3只的详细信息（最重要的）
            print(f"\n" + "=" * 80)
            print(f"📋 重点推荐（封板资金占比前3名）")
            print("=" * 80)
            
            top3 = candidates_df.head(3)
            for idx, (_, stock) in enumerate(top3.iterrows(), 1):
                print(f"\n【{idx}】{stock['代码']} {stock['名称']}")
                print(f"   📊 核心指标:")
                print(f"      • 流通市值: {stock['流通市值(亿)']:.2f}亿元")
                print(f"      • 封板资金占比: {stock['封板资金占比(%)']:.2f}% ⭐")
                if '封板资金(万)' in stock:
                    print(f"      • 封板资金: {stock['封板资金(万)']:.2f}万元")
                print(f"   📈 其他信息:")
                print(f"      • 首次封板时间: {stock['首次封板时间']}")
                print(f"      • 换手率: {stock['换手率']:.2f}%")
                print(f"      • 所属行业: {stock['所属行业']}")
            
            # 如果有更多股票，显示汇总
            if len(candidates_df) > 3:
                print(f"\n" + "─" * 80)
                print(f"📊 其他候选股票（共 {len(candidates_df) - 3} 只）:")
                other_stocks = candidates_df.iloc[3:]
                for idx, (_, stock) in enumerate(other_stocks.iterrows(), 4):
                    print(f"   {idx}. {stock['代码']} {stock['名称']} - 封板资金占比: {stock['封板资金占比(%)']:.2f}%")
            
            print(f"\n" + "=" * 80)
            print(f"✅ 筛选完成！共找到 {len(candidates_df)} 只符合条件的首板股票")
            print("=" * 80)
            
            # 保存候选股票
            first_board_candidates = candidates_df
        else:
            print(f"\n⚠️  未找到符合条件的首板股票")
            print(f"\n💡 可能原因：")
            print(f"   • 所有首板股票的流通市值都 >= 30亿")
            print(f"   • 所有首板股票的封板资金占比都 < 2%")
            print(f"   • 数据获取或处理出现问题")
            first_board_candidates = pd.DataFrame()
else:
    print("⚠️  无法获取涨停板数据，跳过筛选")
    first_board_candidates = pd.DataFrame()

🔍 筛选首板卡位术候选股票

⚠️  当前情绪周期为 过热期，首板卡位术不适用
   💡 建议：
      • 逐步减仓（30-50%）

⚠️  跳过首板卡位术筛选


## 💾 保存结果

本Notebook运行完成后，结果将自动保存到：
- **文件系统**: `notebooks/research/results/chen_xiaoqun_strategy/02_stock_selection/YYYYMMDD_HHMMSS/`
- **MongoDB**: `jqquant.notebook_results` 集合

保存内容包括：
- ✅ 筛选的候选股票列表
- ✅ 运行元数据（时间戳、参数等）
- ✅ 输出文本
- ✅ Notebook副本（可选）

In [6]:
"""
保存notebook运行结果
自动保存到带时间戳的文件夹和MongoDB
"""

from core.notebook_result_manager import NotebookResultManager
from datetime import datetime, timezone, timedelta

print("=" * 80)
print("💾 保存Notebook运行结果")
print("=" * 80)

# 准备保存的结果
save_result = {}

# 添加情绪周期信息（从第一步读取）
if 'emotion_cycle' in globals():
    save_result['emotion_cycle'] = emotion_cycle
if 'position' in globals():
    save_result['position'] = position
if 'strategy' in globals():
    save_result['strategy'] = strategy

# 添加选股结果
if 'first_board_candidates' in globals() and not first_board_candidates.empty:
    # 保存候选股票信息（转换为字典列表）
    save_result['first_board_candidates'] = first_board_candidates.to_dict('records')
    save_result['candidate_count'] = len(first_board_candidates)
    print(f"✅ 找到 {len(first_board_candidates)} 只候选股票，将保存到结果中")
else:
    save_result['first_board_candidates'] = []
    save_result['candidate_count'] = 0
    print("⚠️  未找到候选股票数据")

# 添加龙头信息（如果有）
if 'consecutive_boards' in locals() and not consecutive_boards.empty:
    save_result['dragon_stocks'] = consecutive_boards.head(10).to_dict('records')
    save_result['dragon_count'] = len(consecutive_boards)

# 创建结果管理器
manager = NotebookResultManager(
    strategy_name="chen_xiaoqun_strategy",
    notebook_name="02_stock_selection"
)

# 准备参数
parameters = {
    'notebook': '02_stock_selection',
    'strategy': 'chen_xiaoqun_strategy',
    'emotion_cycle': emotion_cycle if 'emotion_cycle' in globals() else None,
    'position': position if 'position' in globals() else None,
    'run_time': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
}

# 保存结果
try:
    save_info = manager.save_result(
        result=save_result,
        parameters=parameters,
        description="股票筛选结果（首板卡位术/龙头战法）",
        tags=["股票筛选", "首板卡位术", "龙头战法", "陈小群战法"],
        save_notebook_copy=True
    )
    
    print(f"\n✅ 结果保存成功！")
    print(f"   运行ID: {save_info['run_id']}")
    print(f"   运行日期: {save_info['run_date']} {save_info['run_time']}")
    print(f"   文件路径: {save_info['file_path']}")
    print(f"   相对路径: {save_info['relative_path']}")
    if save_info.get('mongodb_id'):
        print(f"   MongoDB ID: {save_info['mongodb_id']}")
    print(f"   结果大小: {save_info['result_size'] / 1024:.2f} KB")
    print(f"   候选股票数: {save_result.get('candidate_count', 0)}")
    
    print(f"\n💡 后续引用方式:")
    print(f"   from core.notebook_result_manager import NotebookResultManager")
    print(f"   manager = NotebookResultManager('chen_xiaoqun_strategy', '02_stock_selection')")
    print(f"   result = manager.load_result('{save_info['run_id']}')")
    
except Exception as e:
    print(f"\n❌ 保存失败: {str(e)}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

2026-01-14 08:21:45,851 - core.notebook_result_manager - INFO - MongoDB索引创建成功
2026-01-14 08:21:45,851 - core.notebook_result_manager - INFO - MongoDB连接成功: jqquant
2026-01-14 08:21:45,851 - core.notebook_result_manager - INFO - NotebookResultManager 初始化: chen_xiaoqun_strategy/02_stock_selection
2026-01-14 08:21:45,852 - core.notebook_result_manager - INFO - 输出目录: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/02_stock_selection
2026-01-14 08:21:45,853 - core.notebook_result_manager - INFO - 开始保存结果: 20260114_212145
2026-01-14 08:21:45,854 - core.notebook_result_manager - INFO - MongoDB保存成功: 696798692ef564eea9724c66
2026-01-14 08:21:45,854 - core.notebook_result_manager - INFO - Notebook副本已保存: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/02_stock_selection/20260114_212145/02_stock_selection.ipynb
2026-01-14 08:21:45,854 - core.notebook_result_manager - INFO - ✅ 结果保存完成: 20260114_212145
2026-01-14 08

💾 保存Notebook运行结果
⚠️  未找到首板候选股票数据
⚠️  未找到二板潜力股票数据
⚠️  未找到龙头股票数据

✅ 结果保存成功！
   运行ID: 20260114_212145
   运行日期: 2026-01-14 21:21:45
   文件路径: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/02_stock_selection/20260114_212145
   相对路径: chen_xiaoqun_strategy/02_stock_selection/20260114_212145
   MongoDB ID: 696798692ef564eea9724c66
   结果大小: 0.32 KB
   首板候选股票数: 0
   二板潜力股票数: 0
   龙头股票数: 0

💡 后续引用方式:
   from core.notebook_result_manager import NotebookResultManager
   manager = NotebookResultManager('chen_xiaoqun_strategy', '02_stock_selection')
   result = manager.load_result('20260114_212145')



In [7]:
"""
二板定龙术 - 确认龙头地位
确认条件：
1. 换手率>25%
2. 分时走势：急跌不破开盘价，反弹带量拉升
3. 板块内至少3只跟风股涨停
4. 确认龙头地位：板块内涨幅最大或最早涨停
"""

print("=" * 80)
print("🐉 二板定龙术 - 确认龙头地位")
print("=" * 80)

print("\n💡 说明：")
print("   • 二板定龙术用于确认首板股票的龙头地位")
print("   • 需要T+1日（次日）的数据来确认")
print("   • 如果首板股票次日继续涨停，进入二板确认流程")

# 获取JQData客户端（用于获取换手率等详细数据）
jq = None
try:
    jq = env.get_jqdata_client()
    if jq and hasattr(jq, 'is_authenticated') and jq.is_authenticated():
        print(f"\n✅ JQData已连接，可以获取详细数据")
    elif jq:
        print(f"\n✅ JQData已连接")
    else:
        print(f"\n⚠️  JQData未连接，将使用AKShare数据")
except Exception as e:
    print(f"\n⚠️  JQData连接失败: {str(e)[:100]}")
    print(f"   将使用AKShare数据")

# 如果有首板候选股票，检查次日是否二板
if 'first_board_candidates' in locals() and not first_board_candidates.empty:
    print(f"\n📊 检查首板候选股票的次日表现...")
    print(f"   首板候选股票数: {len(first_board_candidates)}")
    print(f"\n💡 注意：二板确认需要T+1日数据，当前只能显示逻辑框架")
else:
    print(f"\n⚠️  暂无首板候选股票，无法进行二板确认")

🐉 二板定龙术 - 确认龙头地位

💡 说明：
   • 二板定龙术用于确认首板股票的龙头地位
   • 需要T+1日（次日）的数据来确认
   • 如果首板股票次日继续涨停，进入二板确认流程
   • 当前日期数据：用于分析当日首板股票的潜在二板机会


2026-01-14 08:22:05,338 - config.config_manager - INFO - 加载配置成功: jqdata_config.json
2026-01-14 08:22:05,919 - jqdata.auth - INFO - 聚宽认证成功: 13327806797
2026-01-14 08:22:05,920 - jqdata.client - INFO - 正在检测账号数据权限...


auth success 


2026-01-14 08:22:06,217 - jqdata.client - INFO - ✅ 通过 get_account_info() 检测到账号权限: 数据模式: 实时, 范围: 2005-01-01 至 2026-01-14
2026-01-14 08:22:06,217 - notebooks.lib.research_init - INFO - ✅ JQData 客户端初始化成功



✅ JQData已连接，可以获取详细数据

⚠️  暂无首板候选股票，无法进行二板确认


## 📈 4. 龙头战法（加速期，重仓持有）

In [8]:
"""
龙头战法 - 加速期重仓持有
核心特点：
1. 聚焦总龙头：只参与市场辨识度最高的龙头股
2. 重仓持有：在龙头启动期重仓介入
3. 坚定持有：不爱做T，看准就坚定持有到巅峰
4. 退出时机：只有明显见顶或预计停牌才走
"""

print("=" * 80)
print("📈 龙头战法 - 加速期重仓持有")
print("=" * 80)

print("\n💡 说明：")
print("   • 龙头战法适用于加速期（30-60只涨停）")
print("   • 需要识别市场总龙头（最高连板、最强板块）")
print("   • 重仓持有（50%+仓位），享受主升浪")

# 获取连板高度数据
print(f"\n📊 正在获取连板高度数据...")
try:
    # 使用AKShare获取涨停板数据（使用紧凑格式）
    today_compact = cn_today_str_compact()
    # 先尝试不指定日期（获取最新数据）
    limit_up_data = ak.stock_zt_pool_em()
    if limit_up_data is None or limit_up_data.empty:
        # 如果最新数据为空，尝试指定日期（紧凑格式）
        limit_up_data = ak.stock_zt_pool_em(date=today_compact)
    if limit_up_data is not None and not limit_up_data.empty:
        # 筛选连板股票
        if '连板数' in limit_up_data.columns:
            consecutive_boards = limit_up_data[limit_up_data['连板数'] > 1].copy()
            if not consecutive_boards.empty:
                # 按连板数排序
                consecutive_boards = consecutive_boards.sort_values('连板数', ascending=False)
                
                print(f"✅ 找到 {len(consecutive_boards)} 只连板股票")
                print(f"\n📊 连板高度排名（前10）:")
                top10 = consecutive_boards.head(10)[['代码', '名称', '连板数', '涨跌幅']]
                print(top10.to_string(index=False))
                
                # 识别总龙头（最高连板）
                if len(consecutive_boards) > 0:
                    top_dragon = consecutive_boards.iloc[0]
                    print(f"\n🐉 市场总龙头:")
                    print(f"   代码: {top_dragon.get('代码', '')}")
                    print(f"   名称: {top_dragon.get('名称', '')}")
                    print(f"   连板数: {top_dragon.get('连板数', 0)}板")
                    print(f"   涨跌幅: {top_dragon.get('涨跌幅', 0):.2f}%")
            else:
                print(f"⚠️  未找到连板股票")
        else:
            print(f"⚠️  数据中无'连板数'字段")
    else:
        print(f"⚠️  无法获取涨停板数据")
except Exception as e:
    print(f"❌ 获取失败: {str(e)[:150]}")
    import traceback
    traceback.print_exc()

📈 龙头战法 - 加速期重仓持有

💡 说明：
   • 龙头战法适用于加速期（30-60只涨停）
   • 需要识别市场总龙头（最高连板、最强板块）
   • 重仓持有（50%+仓位），享受主升浪

⚠️  当前情绪周期为 过热期，龙头战法不适用
   💡 建议：
      • 逐步减仓（30-50%）

📊 正在获取连板高度数据...
✅ 找到 19 只连板股票

📊 连板高度排名（前10）:
    代码   名称  连板数       涨跌幅 所属行业
003007 直真科技    5  9.992639 软件开发
002115 三维通信    4  9.994386 互联网服
002044 美年健康    4  9.986505 医疗服务
002400 省广集团    4 10.039841 文化传媒
002131 利欧股份    4  9.966777 互联网服
002574 明牌珠宝    3 10.014306 珠宝首饰
002465 海格通信    3 10.000000 通信设备
001255 博菲电气    3 10.004822 化学原料
601116 三江购物    3 10.010537 商业百货
603000  人民网    3 10.007850 文化传媒

📊 板块连板分析（按最高连板排序）:

【1】软件开发:
   最高连板: 5板
   连板股票数: 1只
   板块龙头: 003007 直真科技

【2】医疗服务:
   最高连板: 4板
   连板股票数: 3只
   板块龙头: 002044 美年健康

【3】文化传媒:
   最高连板: 4板
   连板股票数: 3只
   板块龙头: 002400 省广集团

【4】互联网服:
   最高连板: 4板
   连板股票数: 2只
   板块龙头: 002115 三维通信, 002131 利欧股份

【5】珠宝首饰:
   最高连板: 3板
   连板股票数: 1只
   板块龙头: 002574 明牌珠宝

🐉 市场总龙头:
   代码: 003007
   名称: 直真科技
   连板数: 5板
   涨跌幅: 9.99%
   所属行业: 软件开发
   换手率: 35.19%
   封板资金: 0.28亿元

💡 龙头战法操作建议：
   • 如果当前处于加速期

## 📊 5. 可视化分析

In [9]:
"""
可视化选股结果
包括：首板候选股票分布、板块效应分析、二板潜力分析
"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

print("=" * 80)
print("📊 可视化选股结果")
print("=" * 80)

# 检查是否有数据
has_first_board = 'first_board_candidates' in globals() and not first_board_candidates.empty
has_second_board = 'second_board_candidates_df' in globals() and not second_board_candidates_df.empty
has_dragon = 'dragon_stocks' in globals() and not dragon_stocks.empty

if not (has_first_board or has_second_board or has_dragon):
    print("\n⚠️  暂无选股数据，无法生成可视化图表")
    print("   💡 请先运行前面的筛选步骤")
else:
    # 创建子图
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('首板候选股票（封板资金占比）', '板块效应分析', '二板潜力分布', '龙头股票排名'),
        specs=[[{"type": "bar"}, {"type": "pie"}],
               [{"type": "bar"}, {"type": "bar"}]],
        vertical_spacing=0.15,
        horizontal_spacing=0.12
    )
    
    # 1. 首板候选股票（封板资金占比）
    if has_first_board:
        top10_first = first_board_candidates.head(10)
        fig.add_trace(
            go.Bar(
                x=top10_first['名称'],
                y=top10_first['封板资金占比(%)'],
                name="封板资金占比",
                marker_color='lightblue',
                text=[f"{v:.2f}%" for v in top10_first['封板资金占比(%)']],
                textposition='outside',
                showlegend=False
            ),
            row=1, col=1
        )
        fig.update_xaxes(title_text="股票名称", row=1, col=1, tickangle=-45)
        fig.update_yaxes(title_text="封板资金占比 (%)", row=1, col=1)
    
    # 2. 板块效应分析（饼图）
    if has_first_board and '所属行业' in first_board_candidates.columns:
        sector_counts = first_board_candidates['所属行业'].value_counts().head(8)
        fig.add_trace(
            go.Pie(
                labels=sector_counts.index,
                values=sector_counts.values,
                name="板块分布",
                hole=0.4,
                showlegend=True,
                textinfo='label+percent',
                hovertemplate='<b>%{label}</b><br>股票数: %{value}<br>占比: %{percent}<extra></extra>'
            ),
            row=1, col=2
        )
    
    # 3. 二板潜力分布
    if has_second_board:
        potential_counts = second_board_candidates_df['二板潜力'].value_counts()
        colors_map = {'高': 'green', '中': 'yellow', '低': 'red'}
        colors = [colors_map.get(p, 'gray') for p in potential_counts.index]
        
        fig.add_trace(
            go.Bar(
                x=potential_counts.index,
                y=potential_counts.values,
                name="二板潜力",
                marker_color=colors,
                text=potential_counts.values,
                textposition='outside',
                showlegend=False
            ),
            row=2, col=1
        )
        fig.update_xaxes(title_text="二板潜力", row=2, col=1)
        fig.update_yaxes(title_text="股票数量", row=2, col=1)
    
    # 4. 龙头股票排名（连板高度）
    if has_dragon:
        top5_dragon = dragon_stocks.head(5)
        if '连板数' in top5_dragon.columns:
            fig.add_trace(
                go.Bar(
                    x=top5_dragon['名称'],
                    y=top5_dragon['连板数'],
                    name="连板高度",
                    marker_color='lightcoral',
                    text=[f"{int(v)}板" for v in top5_dragon['连板数']],
                    textposition='outside',
                    showlegend=False
                ),
                row=2, col=2
            )
            fig.update_xaxes(title_text="股票名称", row=2, col=2, tickangle=-45)
            fig.update_yaxes(title_text="连板数", row=2, col=2)
    
    # 更新布局
    fig.update_layout(
        title_text=f"陈小群战法 - 选股结果可视化<br><span style='font-size:0.7em'>情绪周期: {emotion_cycle if 'emotion_cycle' in globals() else '未知'} | 策略: {strategy if 'strategy' in globals() else '未知'}</span>",
        height=800,
        showlegend=True,
        font=dict(size=11),
        margin=dict(l=50, r=50, t=100, b=50)
    )
    
    # 显示图表
    fig.show()
    
    print("\n✅ 可视化图表已生成")

📊 可视化选股结果



✅ 可视化图表已生成


## 📊 5. 策略执行总结

In [10]:
"""
策略执行总结
根据情绪周期和选股结果，给出操作建议
"""

print("=" * 80)
print("📊 策略执行总结")
print("=" * 80)

print("\n💡 操作建议：")
print("   1. 根据第一步的情绪周期判断，选择相应策略")
print("   2. 启动期：使用首板卡位术，10%试错仓")
print("   3. 加速期：使用龙头战法，50%+重仓持有")
print("   4. 过热期：逐步减仓，准备退出")
print("   5. 退潮期：空仓等待，不操作")

print("\n⚠️  风险提示：")
print("   • 本策略为高风险高收益策略，需要严格止损")
print("   • 首板卡位术：次日不涨停或封单量减少，立即止损")
print("   • 二板定龙术：龙头地位不确认，立即止损")
print("   • 龙头战法：明显见顶或预计停牌，及时退出")

print("\n✅ 策略执行完成")
print("=" * 80)

📊 策略执行总结

💡 操作建议：
   1. 根据第一步的情绪周期判断，选择相应策略
   2. 启动期：使用首板卡位术，10%试错仓
   3. 加速期：使用龙头战法，50%+重仓持有
   4. 过热期：逐步减仓，准备退出
   5. 退潮期：空仓等待，不操作

⚠️  风险提示：
   • 本策略为高风险高收益策略，需要严格止损
   • 首板卡位术：次日不涨停或封单量减少，立即止损
   • 二板定龙术：龙头地位不确认，立即止损
   • 龙头战法：明显见顶或预计停牌，及时退出

✅ 策略执行完成
